# INOVAAPPS 2026 · Score de risco e fila de atendimento (motor genérico)

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

from dados import CORES, carregar, fmt_brl
from score import rodar

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates["inova"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, system-ui, sans-serif", size=13, color="#2b2b29"),
    title=dict(font=dict(size=17), x=0.01, xanchor="left"),
    separators=",.", paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    hoverlabel=dict(bgcolor="#ffffff", bordercolor="#c9c7c0", font=dict(size=12, color="#2b2b29")),
    legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="right", x=1, title_text=""),
    margin=dict(l=70, r=30, t=110, b=60),
    xaxis=dict(gridcolor="#eceae5", linecolor="#bdbbb4", zeroline=False),
    yaxis=dict(gridcolor="#eceae5", linecolor="#bdbbb4", zeroline=False),
))
pio.templates.default = "plotly_white+inova"
CINZA, CINZA_CLARO, TINTA = "#8a8984", "#eeede9", "#2b2b29"
AMBAR = "#d68a00"
COR_FAIXA = {"alto": CORES["Cancelou"], "atencao": AMBAR, "baixo": CORES["Ativo"]}
NOME_FAIXA = {"alto": "Risco alto", "atencao": "Atenção", "baixo": "Baixo"}
COR_METODO = {"melhor_variavel": "#b5b3ac", "motor_alto": "#1c5cab", "motor_alto_ou_atencao": "#7aa7dd"}

def rgba(hex_, a):
    h = hex_.lstrip("#")
    return f"rgba({int(h[:2], 16)},{int(h[2:4], 16)},{int(h[4:], 16)},{a})"

def br(v, casas=2):
    return f"{v:.{casas}f}".replace(".", ",").replace("-", "−")

def pct(p, casas=1):
    return br(100 * p, casas) + "%"

def low(s):
    return s[0].lower() + s[1:]

def sub(txt):
    return f"<br><span style='font-size:12px;color:{CINZA}'>{txt}</span>"

d = carregar()
r = rodar()                       # motor genérico: painel → validação aninhada → seleção + modelo final → fila
P, M, F, E, CUT, META = r.painel, r.modelo, r.fila, r.explicacao, r.cortes, r.meta
V, W = r.validacao, r.pesos
PV = pd.DataFrame(W["variaveis"]).set_index("id")
ROT = {k: v.rotulo for k, v in META.items()}
N_CANC, N_ATIV = V["painel"]["cancelados"], V["painel"]["ativos"]
assert len(F) == N_ATIV == 58 and F.prioridade.tolist() == list(range(1, N_ATIV + 1))
print(f"Painel: {V['painel']['linhas']} linhas cliente × mês · {V['painel']['linhas_treino']} de treino "
      f"({V['painel']['positivos_treino']} positivos = cancelou nos {V['horizonte_meses']} meses seguintes; "
      f"{V['painel']['linhas_censuradas']} linhas recentes de ativos fora do treino por censura) · "
      f"{W['resumo']['n_candidatas']} variáveis candidatas → {W['resumo']['n_selecionadas']} selecionadas "
      f"(limite {W['resumo']['limite_variaveis']}) · C final = {br(M.C, 1)}")

Painel: 1295 linhas cliente × mês · 881 de treino (60 positivos = cancelou nos 3 meses seguintes; 174 linhas recentes de ativos fora do treino por censura) · 82 variáveis candidatas → 11 selecionadas (limite 11) · C final = 0,1


## 1. Variáveis que o motor escolheu sozinho

In [2]:
q = PV[PV.frequencia_selecao.notna()].sort_values(["frequencia_selecao", "auc_univariada"], ascending=False).head(28).iloc[::-1]

def grupo(row):
    if row.selecionada:
        return "Selecionada"
    m = str(row.motivo_descarte)
    return "Fora do limite de variáveis" if m.startswith("fora do limite") else "Abaixo de 60% das rodadas" if m.startswith("escolhida") else "Outro motivo"

COR_G = {"Selecionada": "#1c5cab", "Fora do limite de variáveis": "#7aa7dd", "Abaixo de 60% das rodadas": "#c9c7c0", "Outro motivo": "#8a8984"}
q = q.assign(grupo=q.apply(grupo, axis=1))
fig = go.Figure()
for g_, cor in COR_G.items():
    s = q[q.grupo == g_]
    if s.empty:
        continue
    fig.add_trace(go.Bar(x=100 * s.frequencia_selecao, y=s.rotulo, orientation="h", name=g_, marker_color=cor,
                         customdata=np.c_[s.auc_univariada.map(br), s.direcao_texto, s.motivo_descarte.fillna("—")],
                         hovertemplate="<b>%{y}</b><br>Escolhida em %{x:.0f}% das rodadas<br>AUC univariada (out-of-fold): %{customdata[0]}"
                                       "<br>Direção aprendida: %{customdata[1]}<br>Motivo: %{customdata[2]}<extra></extra>"))
fig.add_vline(x=60, line=dict(color=TINTA, dash="dash", width=1.2))
fig.add_annotation(x=60, y=1.0, yref="paper", text="mínimo 60%", showarrow=False, xanchor="left", yanchor="bottom", font=dict(size=11, color=TINTA))
fig.update_xaxes(title_text="% das 50 subamostras de clientes em que a variável foi escolhida (elastic-net)", range=[0, 104], ticksuffix="%")
fig.update_yaxes(tickfont_size=11)
sel = PV[PV.selecionada]
origens = list(dict.fromkeys(low(META[v].rotulo_coluna) for v in sel.index))
fig.update_layout(height=820, margin=dict(l=380, t=130), barmode="overlay",
                  title=f"Stability selection: {len(sel)} de {len(PV)} candidatas entram no modelo, vindas de {len(origens)} colunas: "
                        f"{', '.join(origens[:-1])} e {origens[-1]}"
                        + sub(f"28 candidatas mais escolhidas · limite = mín(positivos/5, cancelados/2) = {W['resumo']['limite_variaveis']} variáveis · "
                              "direção de risco aprendida dos dados, nada definido à mão"))
fig.show()

co = pd.DataFrame(V["coeficientes"]).set_index("variavel")
co = co.reindex(co.coef.abs().sort_values().index)
cor = [CORES["Cancelou"] if c > 0 else CORES["Ativo"] for c in co.coef]
ylab = [f"<b>{r_.rotulo}</b><br><span style='font-size:11px;color:{CINZA}'>{'↑ maior' if r_.direcao > 0 else '↓ menor'} = mais risco · "
        f"em {r_.frequencia_dobras * 100:.0f}% das dobras</span>" for r_ in co.itertuples()]
fig = go.Figure(go.Bar(
    x=co.coef, y=ylab, orientation="h", marker=dict(color=cor, line=dict(color="white", width=1)),
    error_x=dict(type="data", symmetric=False, array=(co.max_dobras - co.coef).clip(lower=0).fillna(0),
                 arrayminus=(co.coef - co.min_dobras).clip(lower=0).fillna(0), color=CINZA, thickness=1.2, width=4),
    text=[f"chances ×{br(np.exp(abs(c)))} por 1 desvio" for c in co.coef], textposition="outside", cliponaxis=False,
    customdata=np.c_[co.rotulo, co.odds_ratio.map(br), co.min_dobras.map(lambda v: br(v) if pd.notna(v) else "—"),
                     co.max_dobras.map(lambda v: br(v) if pd.notna(v) else "—"), (100 * co.frequencia_dobras).round(0)],
    hovertemplate="<b>%{customdata[0]}</b><br>Coeficiente (padronizado): %{x:.2f}<br>Chances × %{customdata[1]} a cada +1 desvio"
                  "<br>Selecionada em %{customdata[4]}% das dobras da validação · faixa %{customdata[2]} a %{customdata[3]}<extra></extra>"))
fig.add_vline(x=0, line=dict(color=TINTA, width=1))
fig.update_xaxes(title_text="Coeficiente da regressão logística L2 (log-odds por +1 desvio-padrão)",
                 range=[min(co.coef.min(), co.min_dobras.min()) - 0.4, max(co.coef.max(), co.max_dobras.max()) + 0.6])
fig.update_layout(height=640, margin=dict(l=420, t=130), showlegend=False,
                  title=f"Peso de cada variável selecionada: {low(co.rotulo.iloc[-1])} e {low(co.rotulo.iloc[-2])} pesam mais"
                        + sub(f"Regressão logística L2 (C = {br(M.C, 2)}) nas selecionadas · vermelho: valor alto aumenta o risco · "
                              "azul: valor baixo aumenta · traço = mín–máx nas dobras em que a variável foi selecionada"))
fig.show()
display(PV[PV.selecionada][["rotulo", "coef", "odds_ratio", "auc_univariada", "frequencia_selecao", "frequencia_dobras", "limiar", "direcao_texto"]]
        .round(3).rename(columns={"rotulo": "Variável", "coef": "Coeficiente", "odds_ratio": "Chances ×", "auc_univariada": "AUC univariada",
                                  "frequencia_selecao": "Frequência (rodadas)", "frequencia_dobras": "Frequência (dobras)",
                                  "limiar": "Limiar de evidência", "direcao_texto": "Direção"}).rename_axis(None))
CATEG = [("vazamento", "Possível vazamento"), ("nulos", "Muitos nulos"), ("constante", "Constante ou quase constante"),
         ("redundante", "Redundante (|Spearman| > 0,9)"), ("escolhida em só", "Escolhida em < 60% das rodadas"),
         ("instável", "Sinal instável entre rodadas"), ("oposta", "Direção oposta à univariada"),
         ("fora do limite", "Fora do limite de variáveis"), ("invertido", "Sinal invertido no modelo final")]
cat = PV[~PV.selecionada].motivo_descarte.map(lambda m: next((n for k, n in CATEG if k in str(m)), "Outro"))
display(cat.value_counts().rename("variáveis descartadas").rename_axis("Motivo").to_frame())

,Variável,Coeficiente,Chances ×,AUC univariada,Frequência (rodadas),Frequência (dobras),Limiar de evidência,Direção
uso_plataforma_pct__ultimo,Uso da plataforma (%) (último),-0.506,0.603,0.908,1.00,1.0,65.00,menor = mais risco
nota_nps__variacao,Nota NPS (variação vs média anterior),-0.408,0.665,0.774,0.96,1.0,-1.50,menor = mais risco
uso_plataforma_pct__ultimo__persistencia,Meses seguidos com uso da plataforma (%) (últi...,0.231,1.260,0.842,0.94,0.7,1.00,maior = mais risco
uso_plataforma_pct__variacao,Uso da plataforma (%) (variação vs média anter...,-0.179,0.836,0.874,0.92,1.0,-8.00,menor = mais risco
tempo_medio_resolucao_h__ultimo,Tempo médio de resolução (h) (último),0.513,1.671,0.820,0.92,0.6,28.00,maior = mais risco
reclamacoes_formais__ultimo,Reclamações formais (último),0.179,1.196,0.718,0.90,0.8,1.00,maior = mais risco
chamados_abertos__tendencia,Chamados abertos (tendência 6m),0.552,1.736,0.780,0.80,0.1,0.20,maior = mais risco
uso_plataforma_pct__tendencia,Uso da plataforma (%) (tendência 6m),-0.220,0.803,0.854,0.78,0.9,-2.00,menor = mais risco
nota_nps__ultimo,Nota NPS (último),-0.103,0.902,0.854,0.76,0.6,5.00,menor = mais risco
reclamacoes_formais__tendencia,Reclamações formais (tendência 6m),0.373,1.452,0.727,0.76,0.0,0.05,maior = mais risco


,variáveis descartadas
Motivo,
Escolhida em < 60% das rodadas,56
"Redundante (|Spearman| > 0,9)",9
Fora do limite de variáveis,5
Constante ou quase constante,1


## 2. Validação aninhada × melhor variável sozinha

In [3]:
MET = {m["id"]: m for m in V["metodos"]}
NOME = {"melhor_variavel": f"Melhor variável sozinha ({low(MET['melhor_variavel']['rotulo'])})",
        "motor_alto": f"Motor: faixa alto (p ≥ {br(CUT['alto'])})",
        "motor_alto_ou_atencao": f"Motor: alto ou atenção (p ≥ {br(CUT['atencao'])})"}
itens = [("canc_k1", "Cancelados detectados<br>em −1", N_CANC), ("canc_k2", "… em −2", N_CANC), ("canc_k3", "… em −3", N_CANC),
         ("ativos_hoje", "Ativos em alarme<br>hoje (jun/2026)", N_ATIV), ("ativos_algum_mes", "Ativos com alarme em<br>algum mês (12 meses)", N_ATIV)]
fig = go.Figure()
for m, r_ in MET.items():
    fig.add_trace(go.Bar(x=[i[1] for i in itens], y=[100 * r_[i[0]] / i[2] for i in itens], name=NOME[m], marker_color=COR_METODO[m],
                         marker_line=dict(color="white", width=1), text=[f"{int(r_[i[0]])}" for i in itens], textposition="outside", cliponaxis=False,
                         customdata=[[int(r_[i[0]]), i[2]] for i in itens],
                         hovertemplate=f"<b>{NOME[m]}</b><br>%{{x}}: %{{customdata[0]}} de %{{customdata[1]}} (%{{y:.0f}}%)<extra></extra>"))
fig.add_vrect(x0=2.5, x1=4.5, fillcolor=CINZA_CLARO, opacity=0.5, line_width=0, layer="below")
fig.add_annotation(x=3.5, y=108, text="alarme falso (menor é melhor)", showarrow=False, font=dict(size=11, color=CINZA))
fig.add_annotation(x=1, y=108, text="detecção (maior é melhor)", showarrow=False, font=dict(size=11, color=CINZA))
a_, b_ = MET["motor_alto_ou_atencao"], MET["melhor_variavel"]
fig.update_yaxes(range=[0, 112], ticksuffix="%", title_text=f"% do grupo (cancelados: {N_CANC} · ativos: {N_ATIV})")
fig.update_layout(barmode="group", bargap=0.2, bargroupgap=0.05, height=500, legend=dict(y=1.08),
                  title=f"Motor (alto + atenção) × melhor variável sozinha: {a_['canc_k2']} × {b_['canc_k2']} cancelados pegos 2 meses antes, "
                        f"{a_['ativos_algum_mes']} × {b_['ativos_algum_mes']} ativos alarmados no ano"
                        + sub(f"Out-of-fold com seleção aninhada: {V['n_dobras']} dobras por cliente, o motor inteiro é refeito sem os clientes avaliados · "
                              "rótulo = nº de clientes"))
fig.show()

AUC = pd.DataFrame(V["auc_por_mes"])
NOME_AUC = {"motor": "Motor (out-of-fold, seleção aninhada)", "melhor_variavel": "Melhor variável sozinha (out-of-fold)"}
COR_AUC = {"motor": COR_METODO["motor_alto"], "melhor_variavel": COR_METODO["melhor_variavel"]}
fig = go.Figure()
fig.add_hline(y=0.5, line=dict(color=CINZA, dash="dot", width=1.5))
for s_, nome in NOME_AUC.items():
    q = AUC[AUC.score == s_]
    fig.add_trace(go.Scatter(x=-q.k, y=q.auc, mode="lines+markers", name=nome, line=dict(color=COR_AUC[s_], width=2.5),
                             marker=dict(size=9, line=dict(color="white", width=1.5)), customdata=np.c_[q.k, q.n_canc, q.n - q.n_canc],
                             hovertemplate=f"<b>{nome}</b><br>Avaliado no mês −%{{customdata[0]}}<br>AUC %{{y:.3f}}"
                                           "<br>%{customdata[1]} cancelados · %{customdata[2]} ativos<extra></extra>"))
a = AUC.pivot(index="k", columns="score", values="auc")
fig.update_xaxes(tickvals=list(range(-6, 0)), ticktext=[f"−{k}" for k in range(6, 0, -1)], title_text="Mês da avaliação (meses antes da saída)")
fig.update_yaxes(range=[0.45, 1.01], tickformat=".2f", title_text="AUC Cancelou × Ativo")
fig.update_layout(height=460, legend=dict(y=1.08),
                  title=f"Separação mês a mês: motor {br(a.motor[1])} / {br(a.motor[2])} / {br(a.motor[3])} em −1/−2/−3 × "
                        f"melhor variável {br(a.melhor_variavel[1])} / {br(a.melhor_variavel[2])} / {br(a.melhor_variavel[3])}"
                        + sub("Um valor por cliente no mês −k · cancelados no próprio −k, ativos em −k a partir de jul/2026 · 0,50 = acaso"))
fig.show()

tab = []
for m, r_ in MET.items():
    tab.append({"Método": NOME[m], **{f"Cancelados em −{k}": f"{int(r_[f'canc_k{k}'])}/{N_CANC}" for k in (1, 2, 3)},
                "Antecedência mediana (meses)": br(r_["antecedencia_mediana"], 1), "Ativos em alarme hoje": f"{int(r_['ativos_hoje'])}/{N_ATIV}",
                "Ativos c/ alarme em algum mês (12m)": f"{int(r_['ativos_algum_mes'])}/{N_ATIV}",
                "Meses em alarme nos ativos (12m)": br(r_["meses_alarme_ativos_pct"], 1) + "%"})
display(pd.DataFrame(tab).set_index("Método"))
TOPO = pd.DataFrame(V["precisao_topo"])
display(TOPO.pivot_table(index="k", columns=["metodo", "top"], values="cancelados")
        .rename(index=lambda k: f"Ranking no mês −{k}").rename_axis(index=None, columns=["Cancelados entre os primeiros", "Top"]))

,Cancelados em −1,Cancelados em −2,Cancelados em −3,Antecedência mediana (meses),Ativos em alarme hoje,Ativos c/ alarme em algum mês (12m),Meses em alarme nos ativos (12m)
Método,,,,,,,
Melhor variável sozinha (uso da plataforma (%) (último)),21/22,15/22,13/22,"3,0",3/58,30/58,"15,2%"
"Motor: faixa alto (p ≥ 0,17)",21/22,15/22,8/22,"2,0",3/58,16/58,"5,9%"
"Motor: alto ou atenção (p ≥ 0,07)",21/22,19/22,13/22,"3,0",6/58,28/58,"9,8%"


Cancelados entre os primeiros melhor_variavel       motor      
Top                                        10    20    10    20
Ranking no mês −1                        10.0  20.0  10.0  18.0
Ranking no mês −2                         8.0  13.0  10.0  17.0
Ranking no mês −3                         9.0  12.0   8.0  15.0

## 3. Cortes das faixas

In [4]:
cv = r.curva
fig = go.Figure()
for k, a_ in [(3, 0.35), (2, 0.65), (1, 1.0)]:
    fig.add_trace(go.Scatter(x=cv.meses_alarme_ativos_pct, y=100 * cv[f"canc_k{k}"] / N_CANC, mode="lines", name=f"Cancelados detectados em −{k}",
                             line=dict(color=rgba(CORES["Cancelou"], a_), width=2.5, shape="hv"),
                             customdata=np.c_[cv.corte, cv[f"canc_k{k}"], cv.ativos_hoje],
                             hovertemplate=f"Corte p ≥ %{{customdata[0]:.2f}}<br>Detecta %{{customdata[1]}} de {N_CANC} em −{k}"
                                           "<br>Meses em alarme nos ativos: %{x:.1f}%<br>Ativos em alarme hoje: %{customdata[2]}<extra></extra>"))
r_ = MET["melhor_variavel"]
for k, a_ in [(3, 0.35), (2, 0.65), (1, 1.0)]:
    fig.add_trace(go.Scatter(x=[r_["meses_alarme_ativos_pct"]], y=[100 * r_[f"canc_k{k}"] / N_CANC], mode="markers",
                             name=NOME["melhor_variavel"], legendgroup="mv", showlegend=k == 1,
                             marker=dict(symbol="diamond", size=12, color=rgba(COR_METODO["melhor_variavel"], a_), line=dict(color="#52514e", width=1.5)),
                             hovertemplate=f"<b>{NOME['melhor_variavel']}</b><br>Detecta {int(r_[f'canc_k{k}'])} de {N_CANC} em −{k}"
                                           "<br>Meses em alarme nos ativos: %{x:.1f}%<extra></extra>"))
for nome, c, cor in [("alto", CUT["alto"], CORES["Cancelou"]), ("atenção", CUT["atencao"], AMBAR)]:
    q = cv.loc[np.isclose(cv.corte, c)].iloc[0]
    fig.add_vline(x=q.meses_alarme_ativos_pct, line=dict(color=cor, dash="dash", width=1.5))
    fig.add_annotation(x=q.meses_alarme_ativos_pct, y=4, xanchor="left", yanchor="bottom", showarrow=False, font=dict(size=12, color=cor),
                       text=f"<b>corte {nome}: p ≥ {br(c)}</b><br>{int(q.canc_k1)}/{int(q.canc_k2)}/{int(q.canc_k3)} cancelados em −1/−2/−3"
                            f"<br>{int(q.ativos_hoje)} ativos em alarme hoje<br>precisão nos meses marcados: {pct(q.precisao_linhas, 0)}")
fig.update_xaxes(title_text="Alarme falso: % dos meses (últimos 12) em que um ativo ficou em alarme", ticksuffix="%", range=[-0.5, 32])
fig.update_yaxes(title_text=f"% dos {N_CANC} cancelados detectados", ticksuffix="%", range=[0, 104])
fig.update_layout(height=560, legend=dict(y=1.06),
                  title="Detecção × alarme falso ao mover o corte de probabilidade: onde ficam as faixas"
                        + sub("Alto = menor corte com ≥ 50% de precisão nos meses-cliente marcados · Atenção = corte de Youden nos meses-cliente "
                              "(sensibilidade − alarme falso) · probabilidades out-of-fold · sem limite de capacidade da equipe"))
fig.show()

## 4. Fila de atendimento (jun/2026)

In [5]:
def evid_txt(c, n=3):
    e = E[(E.cliente == c) & E.dispara].head(n)
    return "<br>".join(f"• {t}: {dt}" for t, dt in zip(e.titulo, e.detalhe)) or "—"

FILL = {"alto": "#fbeaea", "atencao": "#fdf3e1", "baixo": "white"}
cli = d.clientes.set_index("cliente_id").loc[F.index]
cols = {"Prioridade": F.prioridade.astype(str), "Cliente": F.index, "Segmento · plano": [f"{s} · {p}" for s, p in zip(cli.segmento, cli.plano)],
        "Valor mensal": F.valor_mensal.map(fmt_brl), "Probabilidade (3 meses)": F.probabilidade.map(pct),
        "Perda anual esperada": F.perda_anual_esperada.map(fmt_brl), "Faixa": F.faixa_risco.map(NOME_FAIXA),
        "Principais evidências": [evid_txt(c) for c in F.index], "Ação sugerida": F.acao_sugerida}
fig = go.Figure(go.Table(columnwidth=[0.7, 0.7, 1.4, 1, 1, 1.1, 0.8, 4.6, 2.4],
                         header=dict(values=[f"<b>{c}</b>" for c in cols], fill_color="#f3f2ee", align="left", font_size=12),
                         cells=dict(values=list(cols.values()), align="left", font_size=11, height=26,
                                    fill_color=[[FILL[f] for f in F.faixa_risco]] * len(cols))))
nf = F.faixa_risco.value_counts()
em = F[F.faixa_risco != "baixo"]
fig.update_layout(height=2400, margin=dict(l=10, r=10, t=110, b=10),
                  title=f"Fila dos {len(F)} ativos por perda anual esperada: {nf.get('alto', 0)} em risco alto e {nf.get('atencao', 0)} em atenção "
                        f"({fmt_brl(em.valor_mensal.sum())}/mês)"
                        + sub("Prioridade = probabilidade de cancelar nos próximos 3 meses × valor mensal × 12 · "
                              "evidências = variáveis que empurram o risco para cima E passam do limiar aprendido"))
fig.show()

## 5. Risco × valor em jogo

In [6]:
fig = go.Figure()
vx = np.geomspace(2000, 45000, 50)
for perda in (2_000, 10_000, 50_000):
    fig.add_trace(go.Scatter(x=vx, y=perda / (12 * vx), mode="lines", line=dict(color="#d6d4ce", dash="dot", width=1),
                             hoverinfo="skip", showlegend=False))
    fig.add_annotation(x=np.log10(40000), y=np.log10(perda / (12 * 40000)), text=f"perda esperada {fmt_brl(perda)}/ano",
                       showarrow=False, font=dict(size=10, color=CINZA), yanchor="bottom", xanchor="right")
for fx in ("baixo", "atencao", "alto"):
    s = F[F.faixa_risco == fx]
    rot = s.index.where((s.faixa_risco != "baixo") | (s.prioridade <= 10), "")
    fig.add_trace(go.Scatter(x=s.valor_mensal, y=s.probabilidade, mode="markers+text", name=f"{NOME_FAIXA[fx]} ({len(s)})", text=rot,
                             textposition="top center", textfont=dict(size=11, color=COR_FAIXA[fx]),
                             marker=dict(color=rgba(COR_FAIXA[fx], 0.8), size=np.clip(np.sqrt(s.perda_anual_esperada) / 8, 7, 34),
                                         line=dict(color="white", width=1.5)),
                             customdata=np.c_[s.index, s.prioridade, s.perda_anual_esperada.map(fmt_brl), [evid_txt(c, 4) for c in s.index], s.acao_sugerida],
                             hovertemplate="<b>%{customdata[0]}</b> · prioridade %{customdata[1]}<br>R$ %{x:,.0f}/mês · probabilidade %{y:.1%}"
                                           "<br>Perda anual esperada: %{customdata[2]}<br>%{customdata[3]}<br><i>%{customdata[4]}</i><extra></extra>"))
for nome, c, cor in [("alto", CUT["alto"], CORES["Cancelou"]), ("atenção", CUT["atencao"], AMBAR)]:
    fig.add_hline(y=c, line=dict(color=cor, dash="dash", width=1.2))
    fig.add_annotation(x=np.log10(2100), y=np.log10(c), text=f"corte {nome} ({pct(c, 0)})", showarrow=False, xanchor="left", yanchor="bottom",
                       font=dict(size=11, color=cor))
fig.update_xaxes(type="log", title_text="Valor mensal do contrato (R$, escala log)", tickprefix="R$ ", tickformat=",.0f", range=[np.log10(2000), np.log10(45000)])
fig.update_yaxes(type="log", title_text="Probabilidade de cancelar nos próximos 3 meses (escala log)", tickformat=".1%",
                 range=[np.log10(max(F.probabilidade.min() * 0.7, 1e-4)), np.log10(0.8)])
fig.update_layout(height=640, legend=dict(y=1.04),
                  title=f"Risco × valor: {F.index[0]} lidera a fila (probabilidade {pct(F.probabilidade.iloc[0], 0)} em "
                        f"{fmt_brl(F.valor_mensal.iloc[0])}/mês); contratos grandes sobem mesmo com risco menor"
                        + sub("Bolha = perda anual esperada · linhas pontilhadas = mesma perda esperada · rótulo: faixa alto/atenção e 10 primeiros da fila"))
fig.show()

## 6. Por que os 3 primeiros da fila

In [7]:
TOP3 = F.index[:3].tolist()
C3 = M.contribuicoes(r.X.loc[F.loc[TOP3, "linha"]])
C3.index = TOP3
b0 = M.intercepto
sig = lambda z: 1 / (1 + np.exp(-z))
fig = make_subplots(rows=3, cols=1, vertical_spacing=0.13,
                    subplot_titles=[f"<b>{c}</b> · prioridade {F.at[c, 'prioridade']} · {fmt_brl(F.at[c, 'valor_mensal'])}/mês · "
                                    f"probabilidade {pct(F.at[c, 'probabilidade'])} · faixa {NOME_FAIXA[F.at[c, 'faixa_risco']].lower()}" for c in TOP3])
for i, c in enumerate(TOP3, start=1):
    ct = C3.loc[c].reindex(C3.loc[c].abs().sort_values(ascending=False).index)
    ec = E[E.cliente == c].set_index("variavel")
    marca = ["▲ " if ec.at[f, "dispara"] else "" for f in ct.index]
    xs = ["Cliente médio"] + [f"{m}{ROT[f]}" for f, m in zip(ct.index, marca)] + ["Este cliente"]
    fig.add_trace(go.Waterfall(x=xs, y=[b0, *ct.values, 0], measure=["absolute"] + ["relative"] * len(ct) + ["total"],
                               increasing=dict(marker=dict(color=CORES["Cancelou"])), decreasing=dict(marker=dict(color=CORES["Ativo"])),
                               totals=dict(marker=dict(color=TINTA)), connector=dict(line=dict(color="#c9c7c0", width=1)),
                               text=[br(b0)] + [("+" if v > 0 else "") + br(v) for v in ct.values] + [br(b0 + ct.sum())], textposition="outside",
                               customdata=[f"Probabilidade do cliente médio: {pct(sig(b0))}"]
                                          + [f"{ec.at[f, 'titulo']}<br>{ec.at[f, 'detalhe']}" for f in ct.index]
                                          + [f"Probabilidade: {pct(sig(b0 + ct.sum()))}"],
                               hovertemplate="<b>%{x}</b><br>%{customdata}<br>Log-odds: %{y:+.2f}<extra></extra>", showlegend=False), row=i, col=1)
    fig.update_yaxes(title_text="log-odds", row=i, col=1)
fig.update_xaxes(tickangle=-25, tickfont_size=10)
fig.update_layout(height=1400, margin=dict(t=140, b=120),
                  title="De onde vem o risco dos 3 primeiros: contribuição de cada variável sobre o cliente médio"
                        + sub(f"Contribuição = coeficiente × valor padronizado (log-odds) · vermelho aumenta, azul reduz · "
                              f"▲ = evidência (passa do limiar aprendido) · cliente médio = {pct(sig(b0))}"))
fig.show()